# Tahap 9 — Monitoring, Feedback & Continuous Improvement
Simulasi drift nyata: data latih 2024 sebagai acuan, OOT 2025 sebagai "data produksi".

In [1]:
import json

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src import config
from src.data_prep import load_processed
from src.monitor import psi, psi_status, psi_table
from src.reporting import fmt_id, save_figure, save_table, set_plot_style
from src.score import TIERS, assign_tier

set_plot_style()
LBL = config.FEATURE_LABELS
meta = json.loads((config.MODELS / "final_threshold.json").read_text())
model = joblib.load(config.MODELS / "final_model.joblib")
X_train, y_train, _, _ = load_processed("train")
X_test, y_test, _, _ = load_processed("test")
X_oot, y_oot, _, _ = load_processed("oot")

## 1. PSI fitur: train 2024 vs test 2024 (kontrol) dan vs OOT 2025

In [2]:
psi_oot = psi_table(X_train, X_oot)
psi_ctrl = psi_table(X_train, X_test).set_index("fitur")["psi"]
psi_oot["psi_kontrol_train_vs_test"] = psi_oot["fitur"].map(psi_ctrl)
psi_oot["label"] = psi_oot["fitur"].map(LBL)
save_table(psi_oot, "t9_psi_features", decimals=4)

fig, ax = plt.subplots(figsize=(7, 4.5))
pt = psi_oot.iloc[::-1]
XMAX = 0.3
ax.barh(pt["label"], pt["psi"].clip(upper=XMAX), color="#4C72B0")
for i, v in enumerate(pt["psi"]):
    if v > XMAX:
        ax.text(XMAX * 0.98, i, f"PSI {fmt_id(v, 2)} →", va="center", ha="right", color="white", fontsize=8)
ax.set_xlim(0, XMAX)
ax.axvline(config.PSI_WARN, color="#DD8452", linestyle="--", linewidth=1, label="Ambang waspada (0,10)")
ax.axvline(config.PSI_ACT, color="#C44E52", linestyle="--", linewidth=1, label="Ambang tindakan (0,25)")
ax.set_xlabel("PSI data latih 2024 vs OOT 2025")
ax.legend(fontsize=8)
save_figure(fig, "f9_psi")
psi_oot

,fitur,psi,status,psi_kontrol_train_vs_test,label
0,ffiec_msa_md_median_family_income,2.686409,Tindakan,0.000297,Median pendapatan keluarga MSA (USD)
1,loan_amount,0.015953,Stabil,0.001658,Jumlah pinjaman (USD)
2,dti_mid,0.005219,Stabil,0.000555,Rasio utang terhadap pendapatan (%)
3,property_value,0.004154,Stabil,0.000660,Nilai properti (USD)
4,ltv_calc,0.003942,Stabil,0.001963,Rasio pinjaman terhadap nilai properti (%)
5,tract_to_msa_income_percentage,0.003856,Stabil,0.000646,Pendapatan tract terhadap MSA (%)
6,income,0.002596,Stabil,0.000941,Pendapatan (ribu USD)
7,loan_term,0.001827,Stabil,0.000013,Tenor pinjaman (bulan)
8,property_to_income,0.001552,Stabil,0.000698,Rasio nilai properti terhadap pendapatan
9,loan_to_income,0.000677,Stabil,0.001432,Rasio pinjaman terhadap pendapatan


## 2. PSI skor model dan distribusi tier

In [3]:
p_tr, p_te, p_oot = (model.predict_proba(X)[:, 1] for X in (X_train, X_test, X_oot))
tiers = {n: pd.Series(assign_tier(p, meta["t_low"], meta["threshold"])) for n, p in (("train", p_tr), ("test", p_te), ("oot", p_oot))}
score_rows = [
    {"objek": "Skor probabilitas model", "pembanding": "train vs test 2024 (kontrol)", "psi": psi(pd.Series(p_tr), pd.Series(p_te))},
    {"objek": "Skor probabilitas model", "pembanding": "train 2024 vs OOT 2025", "psi": psi(pd.Series(p_tr), pd.Series(p_oot))},
    {"objek": "Distribusi risk tier", "pembanding": "train vs test 2024 (kontrol)", "psi": psi(tiers["train"], tiers["test"], categorical=True)},
    {"objek": "Distribusi risk tier", "pembanding": "train 2024 vs OOT 2025", "psi": psi(tiers["train"], tiers["oot"], categorical=True)},
]
score_psi = pd.DataFrame(score_rows)
score_psi["status"] = score_psi["psi"].map(psi_status)
for n in ("train", "oot"):
    vc = tiers[n].value_counts(normalize=True).reindex(TIERS, fill_value=0) * 100
    for t in TIERS:
        score_psi.loc[len(score_psi)] = {"objek": f"Porsi tier {t} (%)", "pembanding": n, "psi": vc[t], "status": "-"}
save_table(score_psi, "t9_psi_score", decimals=4)
score_psi

,objek,pembanding,psi,status
0,Skor probabilitas model,train vs test 2024 (kontrol),0.001120,Stabil
1,Skor probabilitas model,train 2024 vs OOT 2025,0.004184,Stabil
2,Distribusi risk tier,train vs test 2024 (kontrol),0.000039,Stabil
3,Distribusi risk tier,train 2024 vs OOT 2025,0.002203,Stabil
4,Porsi tier Rendah (%),train,70.107498,-
5,Porsi tier Sedang (%),train,17.156428,-
6,Porsi tier Tinggi (%),train,12.736074,-
7,Porsi tier Rendah (%),oot,71.320974,-
8,Porsi tier Sedang (%),oot,17.465863,-
9,Porsi tier Tinggi (%),oot,11.213163,-


## 3. Indikasi model drift: test 2024 vs OOT 2025 (dari Tahap 7)

In [4]:
oot_perf = pd.read_csv(config.TABLES / "t7_oot_performance.csv")
oot_perf[["data", "proporsi_ditolak", "roc_auc", "pr_auc", "recall", "precision", "cost_per_1000"]]

,data,proporsi_ditolak,roc_auc,pr_auc,recall,precision,cost_per_1000
0,Test 2024,0.117535,0.907003,0.789935,0.759474,0.692037,181.074448
1,OOT 2025,0.096730,0.890896,0.728816,0.717864,0.619263,179.147649


## 4. Analisis error pada test (threshold final)

In [5]:
imp = pd.read_csv(config.TABLES / "t7_importance_comparison.csv").sort_values("porsi_shap", ascending=False)
top = [f for f in imp["fitur"] if f in config.NUMERIC_FEATURES][:5]
pred = (p_te >= meta["threshold"]).astype(int)
group = np.select([(y_test == 1) & (pred == 1), (y_test == 1) & (pred == 0), (y_test == 0) & (pred == 1), (y_test == 0) & (pred == 0)],
                  ["TP (ditolak, terdeteksi)", "FN (ditolak, lolos)", "FP (disetujui, ditandai)", "TN (disetujui, aman)"], default="-")
err = X_test[top].groupby(group).median()
err.insert(0, "n", pd.Series(group).value_counts())
err = err.rename(columns=LBL).reset_index(names="kelompok")
save_table(err, "t9_error_analysis", decimals=2)
err

,kelompok,n,Rasio utang terhadap pendapatan (%),Nilai properti (USD),Tenor pinjaman (bulan),Rasio pinjaman terhadap nilai properti (%),Jumlah pinjaman (USD)
0,"FN (ditolak, lolos)",311,42.0,535000.0,360.0,80.121474,415000.0
1,"FP (disetujui, ditandai)",437,46.0,195000.0,360.0,91.304348,175000.0
2,"TN (disetujui, aman)",9271,40.0,595000.0,360.0,80.110497,465000.0
3,"TP (ditolak, terdeteksi)",982,55.0,145000.0,300.0,96.078431,145000.0
